In [5]:
import requests
# 1. 构建请求
request = requests.Request(
    method='GET',
    url='https://movie.douban.com/top250',
    headers={'User-Agent': 'my-bot'}
)
# 2. 准备请求
prepared = request.prepare()  # 生成可发送的请求对象
# 3. 发送请求（底层使用urllib3）
response = requests.Session().send(prepared)
# 4. 处理响应
print(response.status_code)  # HTTP状态码
print(response.headers)     # 响应头
print(response.text)        # 响应体（自动解码）

200
{'Date': 'Sun, 06 Apr 2025 07:01:10 GMT', 'Content-Type': 'text/html; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Keep-Alive': 'timeout=30', 'X-Xss-Protection': '1; mode=block', 'X-Douban-Mobileapp': '0', 'Expires': 'Sun, 1 Jan 2006 01:00:00 GMT', 'Pragma': 'no-cache', 'Cache-Control': 'must-revalidate, no-cache, private', 'X-DAE-App': 'movie', 'X-DAE-Instance': 'default', 'Set-Cookie': 'bid=cq-E139ORBM; Expires=Mon, 06-Apr-26 07:01:10 GMT; Domain=.douban.com; Path=/', 'X-DOUBAN-NEWBID': 'cq-E139ORBM', 'Server': 'dae', 'X-Content-Type-Options': 'nosniff'}
<!DOCTYPE html>
<html lang="zh-CN" class="">
<head>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8">
    <meta name="renderer" content="webkit">
    <meta name="referrer" content="always">
    <meta name="google-site-verification" content="ok0wCgT20tBBgo9_zat2iAcimtN4Ftf5ccsh092Xeyw" />
    <title>
豆瓣电影 Top 250
</title>
    
    <meta name="baidu-site-verification" content="c

SyntaxError: invalid character '（' (U+FF08) (1185262345.py, line 16)

In [36]:
import requests
from bs4 import BeautifulSoup
import time
import csv

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def scrape_douban_top250():
    movies = []
    
    for page in range(0, 250, 25):
        url = f'https://movie.douban.com/top250?start={page}'
        print(f'正在抓取: {url}')
        
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f'请求失败: {e}')
            continue
            
        soup = BeautifulSoup(response.text, 'html.parser')
        items = soup.find_all('div', class_='item')
        
        for item in items:
            try:
                rank = item.find('em').text if item.find('em') else "无排名"
                title = item.find('span', class_='title').text if item.find('span', class_='title') else "无标题"
                rating = item.find('span', class_='rating_num').text if item.find('span', class_='rating_num') else "0.0"
                
                # 更稳定的评价人数提取
                review_span = item.select_one('div.star > span:last-child')
                num_reviews = review_span.text.replace('人评价', '') if review_span else "0"
                
                quote_tag = item.find('span', class_='inq')
                quote = quote_tag.text if quote_tag else "无经典台词"
                
                movies.append({
                    '排名': rank,
                    '名称': title,
                    '评分': rating,
                    '评价人数': num_reviews,
                    '经典台词': quote
                })
            except Exception as e:
                print(f"解析电影条目时出错: {e}")
                continue
        
        time.sleep(2)
    
    # 保存数据
    if movies:  # 只有movies不为空时才保存
        with open('douban_top250.csv', 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=movies[0].keys())
            writer.writeheader()
            writer.writerows(movies)
    
    print(f'抓取完成，共{len(movies)}部电影')

if __name__ == '__main__':
    scrape_douban_top250()

正在抓取: https://movie.douban.com/top250?start=0
正在抓取: https://movie.douban.com/top250?start=25
正在抓取: https://movie.douban.com/top250?start=50
正在抓取: https://movie.douban.com/top250?start=75
正在抓取: https://movie.douban.com/top250?start=100
正在抓取: https://movie.douban.com/top250?start=125
正在抓取: https://movie.douban.com/top250?start=150
正在抓取: https://movie.douban.com/top250?start=175
正在抓取: https://movie.douban.com/top250?start=200
正在抓取: https://movie.douban.com/top250?start=225
抓取完成，共250部电影


In [1]:
import requests
from bs4 import BeautifulSoup
import time
import csv

# 设置请求头，模拟浏览器访问，避免被网站识别为爬虫
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def get_page_content(url):
    """
    发送请求获取网页内容，并进行异常处理
    :param url: 要请求的网页URL
    :return: 网页的文本内容，如果请求失败则返回None
    """
    try:
        # 发送请求，设置超时时间为10秒
        response = requests.get(url, headers=headers, timeout=10)
        # 检查响应状态码，如果不是200则抛出异常
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f'请求失败: {e}')
        return None

def parse_movie_item(item):
    """
    解析单个电影条目的信息
    :param item: 单个电影条目的HTML元素
    :return: 包含电影信息的字典
    """
    try:
        # 提取电影排名
        rank = item.find('em').text if item.find('em') else "无排名"
        # 提取电影标题
        title = item.find('span', class_='title').text if item.find('span', class_='title') else "无标题"
        # 提取电影评分
        rating = item.find('span', class_='rating_num').text if item.find('span', class_='rating_num') else "0.0"
        # 提取评价人数
        review_span = item.select_one('div.star > span:last-child')
        num_reviews = review_span.text.replace('人评价', '') if review_span else "0"
        # 提取经典台词
        quote_tag = item.find('span', class_='inq')
        quote = quote_tag.text if quote_tag else "无经典台词"

        return {
            '排名': rank,
            '名称': title,
            '评分': rating,
            '评价人数': num_reviews,
            '经典台词': quote
        }
    except Exception as e:
        print(f"解析电影条目时出错: {e}")
        return None

def scrape_douban_top250():
    """
    抓取豆瓣电影Top250的信息
    :return: 包含所有电影信息的列表
    """
    movies = []
    # 遍历10页，每页25条记录
    for page in range(0, 250, 25):
        url = f'https://movie.douban.com/top250?start={page}'
        print(f'正在抓取: {url}')
        # 获取当前页的网页内容
        page_content = get_page_content(url)
        if page_content is None:
            continue
        # 解析网页内容
        soup = BeautifulSoup(page_content, 'html.parser')
        items = soup.find_all('div', class_='item')
        # 解析每个电影条目
        for item in items:
            movie_info = parse_movie_item(item)
            if movie_info:
                movies.append(movie_info)
        # 每次请求后休眠2秒，避免对网站造成过大压力
        time.sleep(2)
    return movies

def save_to_csv(movies):
    """
    将电影信息保存到CSV文件中
    :param movies: 包含所有电影信息的列表
    """
    if movies:
        with open('douban_top250.csv', 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=movies[0].keys())
            writer.writeheader()
            writer.writerows(movies)
        print(f'抓取完成，共{len(movies)}部电影')
    else:
        print('未抓取到任何电影信息')

if __name__ == '__main__':
    # 抓取电影信息
    movies = scrape_douban_top250()
    # 保存电影信息到CSV文件
    save_to_csv(movies)


正在抓取: https://movie.douban.com/top250?start=0
请求失败: 403 Client Error: Forbidden for url: https://sec.douban.com/b?r=https%3A%2F%2Fmovie.douban.com%2Ftop250%3Fstart%3D0
正在抓取: https://movie.douban.com/top250?start=25
请求失败: 403 Client Error: Forbidden for url: https://sec.douban.com/b?r=https%3A%2F%2Fmovie.douban.com%2Ftop250%3Fstart%3D25
正在抓取: https://movie.douban.com/top250?start=50
请求失败: 403 Client Error: Forbidden for url: https://sec.douban.com/b?r=https%3A%2F%2Fmovie.douban.com%2Ftop250%3Fstart%3D50
正在抓取: https://movie.douban.com/top250?start=75
请求失败: 403 Client Error: Forbidden for url: https://sec.douban.com/b?r=https%3A%2F%2Fmovie.douban.com%2Ftop250%3Fstart%3D75
正在抓取: https://movie.douban.com/top250?start=100
请求失败: 403 Client Error: Forbidden for url: https://sec.douban.com/b?r=https%3A%2F%2Fmovie.douban.com%2Ftop250%3Fstart%3D100
正在抓取: https://movie.douban.com/top250?start=125
请求失败: 403 Client Error: Forbidden for url: https://sec.douban.com/b?r=https%3A%2F%2Fmovie.douban.com%2